<a href="https://colab.research.google.com/github/ankorn/arcanumsearch/blob/main/arc_synth.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import random
import re
import json
from typing import List, Dict, Optional
from dataclasses import dataclass
import time
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

In [3]:
model_name = "Qwen/Qwen2.5-14B-Instruct" # "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
            model_name,
            trust_remote_code=True
        )
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/579 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [39]:
@dataclass
class QueryExample:
    query: str
    query_type: str  # 'informational', 'navigational', 'transactional', 'vague'
    difficulty: str  # 'easy', 'medium', 'hard'

class LLMQueryGenerator:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    def _generate(self, prompt: str, max_tokens: int = 512, temperature: float = 0.8) -> str:
        inputs = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)

        outputs = self.model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            do_sample=True,
            top_p=0.9,
            pad_token_id=self.tokenizer.eos_token_id,
            stop_strings=["]"],
            tokenizer=self.tokenizer
        )

        response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
        response = response[len(prompt):].strip()
        return response

    def _create_query_generation_prompt(self,
                                        document_title: str,
                                        document_text: str,
                                        num_queries: int = 5) -> str:

        return f"""You are a player of the RPG game "Arcanum: Of Steamworks and Magick Obscura".
You need help with a quest and are searching a wiki for information.

Given this quest information:
Content: {document_text}...

Generate {num_queries} different search queries that a player might type to find this information.
Make them diverse in style and specificity:

Requirements:
1. Mix of specific (quest name) and vague (problem description) queries
2. Include natural language questions and keyword searches
3. Some should be short (2-3 words), others longer full sentences
4. Include common typos or variations a player might use
5. Some should focus on main result of a quest, others on walkthrough steps
6. Avoid using words "arcanum", "experience", "reward"

Output format: Return ONLY a JSON array of {num_queries} strings, like:
["query 1", "query 2", "query 3"]

Queries:"""

    def generate_queries_for_document(self,
                                      title: str,
                                      text: str,
                                      num_queries: int = 3) -> List[QueryExample]:
        prompt = self._create_query_generation_prompt(title, text, num_queries)
        response = self._generate(prompt, temperature=0.9)

        try:
            # Parse JSON response
            queries = json.loads(response)
            if not isinstance(queries, list):
                queries = [str(queries)]
        except json.JSONDecodeError:
            print('>>>', 'json error')
            queries = []

        examples = []
        for i, q in enumerate(queries):
            q_lower = q.lower()
            if any(w in q_lower for w in ['how', 'what', 'where', 'who', 'why']):
                q_type = 'informational'
            elif any(w in q_lower for w in ['find', 'get', 'obtain', 'kill', 'deliver']):
                q_type = 'transactional'
            else:
                q_type = 'navigational'

            # Estimate difficulty
            if len(q.split()) <= 3:
                difficulty = 'easy'
            elif any(w in q_lower for w in title.lower().split()):
                difficulty = 'easy'
            else:
                difficulty = 'medium'

            examples.append(QueryExample(
                query=q,
                query_type=q_type,
                difficulty=difficulty,
            ))
            
        return examples

class ArcanumSyntheticDataset:
    def __init__(self, df, llm_generator: Optional[LLMQueryGenerator] = None):
        self.df = df
        self.generator = llm_generator

    def generate(self,
                 queries_per_doc: int = 3) -> pd.DataFrame:

        results = []

        for idx, row in self.df.iterrows():
            print(f"Processing {idx+1}/{len(self.df)}: {row['document_title'][:50]}...")

            if self.generator:
                examples = self.generator.generate_queries_for_document(
                    row['document_title'], row['enhanced_document_text'], queries_per_doc
                )
                for ex in examples:
                    results.append({
                        'query': ex.query,
                        'document_title': row['document_title'],
                        'document_text': row['document_text'],
                        'enhanced_document_text': row['enhanced_document_text'],
                        'document_link': row['document_link'],
                        'query_type': ex.query_type,
                        'difficulty': ex.difficulty
                    })

        df = pd.DataFrame(results)
        print(f"\nGenerated {len(df)} examples")
        return df

    def _create_row(self, row, query: str, label: int, label_type: str) -> Dict:
        return {
            'query': query,
            'document_title': row['document_title'],
            'document_text': row['document_text'],
            'document_link': row['document_link'],
            'label': label,
            'label_type': label_type
        }

In [5]:
from datasets import load_dataset

init_dataset = load_dataset("pameydorke/arcanum-quests-queries-synthetic")

README.md:   0%|          | 0.00/625 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/215k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/122k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/549 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/138 [00:00<?, ? examples/s]

In [40]:
pure_df_train = init_dataset['train'].to_pandas()[['document_title', 'document_link', 'enhanced_document_text', 'document_text']]
pure_df_test = init_dataset['test'].to_pandas()[['document_title', 'document_link', 'enhanced_document_text', 'document_text']]

pure_df = pd.concat([pure_df_train, pure_df_test]).drop_duplicates(subset=['document_title'])

pure_df = pure_df.reset_index()

pure_df = pure_df.drop(columns=['index'])

pure_df

,document_title,document_link,enhanced_document_text,document_text
0,The Main Quest: Part 01 - Finding out about th...,https://arcanum.fandom.com/wiki/The_Main_Quest,The Main Quest. Finding out about the ring. So...,"So, you've got this ring you got from a gnome,..."
1,The Sorcerous Beast: Short Description,https://arcanum.fandom.com/wiki/The_Sorcerous_...,The Sorcerous Beast. From talking with the gua...,From talking with the guard outside the isle o...
2,Find Liam Cameron: Short Description,https://arcanum.fandom.com/wiki/Find_Liam_Cameron,Find Liam Cameron. 1 - Starting Point2 - Liam'...,1 - Starting Point2 - Liam's WorkshopThere are...
3,Collect from Larrs: Walkthrough,https://arcanum.fandom.com/wiki/Collect_from_L...,Collect from Larrs. The Pub If the player is i...,The Pub If the player is interested in joining...
4,"Ancient Gods: Altars, Offerings, and Blessings",https://arcanum.fandom.com/wiki/Ancient_Gods,"Ancient Gods. Altars, Offerings, and Blessings...","Each god has an altar, and each altar can acce..."
...,...,...,...,...
224,Clear the Mines: Short Description,https://arcanum.fandom.com/wiki/Clear_the_Mines,"Clear the Mines. Arvid Millstone, the foreman ...","Arvid Millstone, the foreman of the miners in ..."
225,The Ren'ar Siamese Twins: Looking for the Skul...,https://arcanum.fandom.com/wiki/The_Ren%27ar_S...,The Ren'ar Siamese Twins. Looking for the Skul...,(OPTIONAL) The quest begins with Benjamin Gers...
226,Defeat Kerghan: Short Description,https://arcanum.fandom.com/wiki/Defeat_Kerghan,Defeat Kerghan. 143 Arronax: Defeat Kerghan. (...,143 Arronax: Defeat Kerghan. (Prereq: Q142. Re...
227,Masters of Magick: Master Of Meta,https://arcanum.fandom.com/wiki/Masters_of_Magick,Masters of Magick. Master Of Meta. You can bec...,You can become the Master Of Meta by talking w...


In [41]:
generator = LLMQueryGenerator(
    model,
    tokenizer
)

dataset = ArcanumSyntheticDataset(pure_df, generator)
df = dataset.generate(
    queries_per_doc=5,
)

df

Processing 1/229: The Main Quest: Part 01 - Finding out about the ri...
Processing 2/229: The Sorcerous Beast: Short Description...
Processing 3/229: Find Liam Cameron: Short Description...
Processing 4/229: Collect from Larrs: Walkthrough...
Processing 5/229: Ancient Gods: Altars, Offerings, and Blessings...
Processing 6/229: Kill Damian Maug: Background...
Processing 7/229: Ancient Gods: Multiple Offerings...
Processing 8/229: Investigate M'in Gorad: Short Description...
Processing 9/229: Negotiations with Caladon: Notes...
Processing 10/229: Find Dudley Crosston: Notes...
Processing 11/229: Masters of Magick: Master Of Phantasm...
Processing 12/229: Meet with Vollinger: Walkthrough...
Processing 13/229: Meet with Vollinger: Notes...
Processing 14/229: Remove the Humans from Falcon's Ache: Short Descri...
Processing 15/229: Get Mithril Ore from the Wheel Clan: Short Descrip...
Processing 16/229: Fix the Town's Steam Engine: Rewards...
Processing 17/229: Masters of Magick: Master Of C

,query,document_title,document_text,enhanced_document_text,document_link,query_type,difficulty
0,where is p schuyler sons,The Main Quest: Part 01 - Finding out about th...,"So, you've got this ring you got from a gnome,...",The Main Quest. Finding out about the ring. So...,https://arcanum.fandom.com/wiki/The_Main_Quest,informational,medium
1,ring quest arcanaum,The Main Quest: Part 01 - Finding out about th...,"So, you've got this ring you got from a gnome,...",The Main Quest. Finding out about the ring. So...,https://arcanum.fandom.com/wiki/The_Main_Quest,navigational,easy
2,talk to ristezze about ring,The Main Quest: Part 01 - Finding out about th...,"So, you've got this ring you got from a gnome,...",The Main Quest. Finding out about the ring. So...,https://arcanum.fandom.com/wiki/The_Main_Quest,navigational,easy
3,find out ring info in arcana,The Main Quest: Part 01 - Finding out about th...,"So, you've got this ring you got from a gnome,...",The Main Quest. Finding out about the ring. So...,https://arcanum.fandom.com/wiki/The_Main_Quest,transactional,easy
4,crash site items needed ring quest,The Main Quest: Part 01 - Finding out about th...,"So, you've got this ring you got from a gnome,...",The Main Quest. Finding out about the ring. So...,https://arcanum.fandom.com/wiki/The_Main_Quest,navigational,easy
...,...,...,...,...,...,...,...
1140,how clear halfling gang renzo,Clear the Halfling Gang: Renzo path,"Rather than planting the statue, you can inste...",Clear the Halfling Gang. Renzo path. Rather th...,https://arcanum.fandom.com/wiki/Clear_the_Half...,informational,easy
1141,where find halflings hideout arcu,Clear the Halfling Gang: Renzo path,"Rather than planting the statue, you can inste...",Clear the Halfling Gang. Renzo path. Rather th...,https://arcanum.fandom.com/wiki/Clear_the_Half...,informational,easy
1142,give captain of guard statie location,Clear the Halfling Gang: Renzo path,"Rather than planting the statue, you can inste...",Clear the Halfling Gang. Renzo path. Rather th...,https://arcanum.fandom.com/wiki/Clear_the_Half...,navigational,medium
1143,what do after halflings hideout locaion given,Clear the Halfling Gang: Renzo path,"Rather than planting the statue, you can inste...",Clear the Halfling Gang. Renzo path. Rather th...,https://arcanum.fandom.com/wiki/Clear_the_Half...,informational,easy


In [45]:
import huggingface_hub
huggingface_hub.login()

In [44]:
from datasets import Dataset

Dataset.from_pandas(df).push_to_hub("pameydorke/arcanum-quests-queries-synthetic-v2")

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              : 100%|##########|  245kB /  245kB            

README.md:   0%|          | 0.00/477 [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/pameydorke/arcanum-quests-queries-synthetic-v2/commit/08979b97e6e159fb57101b04d0d6acb91b1ac955', commit_message='Upload dataset', commit_description='', oid='08979b97e6e159fb57101b04d0d6acb91b1ac955', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/pameydorke/arcanum-quests-queries-synthetic-v2', endpoint='https://huggingface.co', repo_type='dataset', repo_id='pameydorke/arcanum-quests-queries-synthetic-v2'), pr_revision=None, pr_num=None)